# Propuesta de asignaturas y NRC a priorizar

Este notebook materializa la logica pedida sobre `Prediccion_final_XGB`:

- Convierte la prediccion final a numerico, corrigiendo valores con coma decimal.
- Crea `Estado XGBOOST` con la regla `Retiro / Perdida / Aprobacion`.
- Marca las asignaturas a ignorar.
- Calcula repitencia por `Descripcion_Materia` y por `nrc`.
- Permite recomendar cobertura de `Asignatura`, de `NRC` o de `Ambas`.
- Genera un grafico textual ASCII con los tres grupos finales.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from propuesta_asignaturas_nrc import (
    DEFAULT_INPUT,
    DEFAULT_OUTPUT_PREFIX,
    build_proposal,
    resolve_output_dir,
    save_outputs,
)

input_csv = Path(DEFAULT_INPUT)
top_n = 100
std_factor = 1.0
subject_high_std_factor = 1.0
output_dir = resolve_output_dir(input_csv, None)

input_csv, output_dir, std_factor, subject_high_std_factor


(WindowsPath('Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv'),
 WindowsPath('Resultados_Modelo_v2/propuestas_resultados_v2'),
 1.0,
 1.0)

In [2]:
outputs = build_proposal(
    input_csv=input_csv,
    top_n=top_n,
    std_factor=std_factor,
    subject_high_std_factor=subject_high_std_factor,
)

enriched_df = outputs["enriched_df"]
subject_summary = outputs["subject_summary"]
nrc_summary = outputs["nrc_summary"]
proposal = outputs["proposal"]
proposal_actions = outputs["proposal_actions"]
flow_visual = outputs["flow_visual"]
text_visual = outputs["text_visual"]

print(flow_visual)
print(f"Filas totales: {len(enriched_df):,}")
print(f"Asignaturas no ignoradas: {subject_summary['Descripcion_Materia'].nunique():,}")
print(f"Asignaturas con repitencia > 0: {(proposal['repitencia_asignatura'] > 0).sum():,}")
print(f"Seleccionadas en top_n: {proposal['seleccion_top_n'].sum():,}")
print(f"Acciones recomendadas: {len(proposal_actions):,}")
display(proposal.loc[proposal['seleccion_top_n'], ['cobertura_recomendada']].value_counts().rename('conteo').reset_index())


FLUJO DEL PROCESO - PROPUESTA DE COBERTURA

[CSV DE ENTRADA]
  Resultados_Modelo_v2\resultados_asig_xg_cat2026-02-23171813.csv
  filas=21607

        |
        v
[1] NORMALIZAR Prediccion_final_XGB
  - convertir coma decimal a punto
  - crear Prediccion_final_XGB_num

        |
        v
[2] CLASIFICAR Estado XGBOOST
  - Retiro si prediccion < 0
  - Perdida si 0 <= prediccion < 3
  - Aprobacion si prediccion >= 3

        |
        v
[3] MARCAR ASIGNATURAS A IGNORAR
  filas_ignoradas=1181
  filas_utiles=20426

        |
        v
[4] CALCULAR REPITENCIA
  asignaturas_utiles=172
  asignaturas_con_repitencia=123
  nrc_unicos=794

        |
        +-------------------------------+
        v                               v
[5A] REPITENCIA ALTA DE ASIGNATURA     [5B] NRC SOBRESALIENTE
  umbral = media + 1 * desv
  umbral = media + 1 * desv dentro de la asignatura
  asignaturas_altas=11
  nrc_sobresalientes=94

        \                               /
         \                            

,cobertura_recomendada,conteo
0,NRC,50
1,Asignatura,44
2,Ambas,6


In [3]:
estado_cols = [
    'Prediccion_final_XGB',
    'Prediccion_final_XGB_num',
    'Estado XGBOOST',
    'Asignatura a ignorar',
]

display(enriched_df[estado_cols].head(10))
display(enriched_df['Estado XGBOOST'].value_counts(dropna=False).rename_axis('Estado').reset_index(name='conteo'))


,Prediccion_final_XGB,Prediccion_final_XGB_num,Estado XGBOOST,Asignatura a ignorar
0,"3,94",3.94,Aprobacion,False
1,"3,22",3.22,Aprobacion,False
2,"3,74",3.74,Aprobacion,False
3,"3,96",3.96,Aprobacion,False
4,"3,45",3.45,Aprobacion,False
5,"3,54",3.54,Aprobacion,False
6,"3,81",3.81,Aprobacion,False
7,"4,38",4.38,Aprobacion,False
8,"3,01",3.01,Aprobacion,False
9,"3,88",3.88,Aprobacion,False


,Estado,conteo
0,Aprobacion,19930
1,Perdida,916
2,Retiro,761


In [4]:
cols_propuesta = [
    'rank_prioridad',
    'Descripcion_Materia',
    'cobertura_recomendada',
    'asignatura_alta_repitencia',
    'cubrir_asignatura',
    'cubrir_nrc',
    'nrc_priorizado',
    'repitencia_asignatura',
    'tasa_repitencia_asignatura',
    'repitencia_nrc_priorizado',
    'tasa_repitencia_nrc_priorizado',
    'criterio_cobertura_asignatura',
    'criterio_cobertura_nrc',
]

display(proposal.loc[proposal['seleccion_top_n'], cols_propuesta].head(top_n))


,rank_prioridad,Descripcion_Materia,cobertura_recomendada,asignatura_alta_repitencia,cubrir_asignatura,cubrir_nrc,nrc_priorizado,repitencia_asignatura,tasa_repitencia_asignatura,repitencia_nrc_priorizado,tasa_repitencia_nrc_priorizado,criterio_cobertura_asignatura,criterio_cobertura_nrc
0,1,ALGORITMIA Y PROGRAMACION I,Ambas,True,True,True,4027.0,175,0.603448,26.0,0.928571,Asignatura con repitencia general superior a m...,NRC con repitencia superior a media + 1 desvia...
1,2,ANÁLISIS DE DATOS EN INGEN I,Ambas,True,True,True,2017.0,64,0.257028,14.0,0.437500,Asignatura con repitencia general superior a m...,NRC con repitencia superior a media + 1 desvia...
2,3,REFUERZO ESTATICA,Ambas,True,True,True,4121.0,55,0.591398,29.0,1.000000,Asignatura con repitencia general superior a m...,NRC con repitencia superior a media + 1 desvia...
3,4,CALCULO I,Ambas,True,True,True,2629.0,55,0.255814,19.0,0.513514,Asignatura con repitencia general superior a m...,NRC con repitencia superior a media + 1 desvia...
4,5,OBLIGACIONES I,Asignatura,True,True,False,NaN,51,0.689189,NaN,NaN,Asignatura con repitencia general superior a m...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,GEOGRAFIA GRAN CUENCA CARIBE,Asignatura,False,True,False,NaN,2,0.021277,NaN,NaN,Cobertura general de la asignatura sin NRC sob...,<NA>
96,97,ESTADISTICA INFERENCIAL,Asignatura,False,True,False,NaN,2,0.020202,NaN,NaN,Cobertura general de la asignatura sin NRC sob...,<NA>
97,98,SOCIOLOGÍA JURÍDICA,Asignatura,False,True,False,NaN,2,0.019608,NaN,NaN,Cobertura general de la asignatura sin NRC sob...,<NA>
98,99,PROCESO ADMINISTRATIVO,NRC,False,False,True,1572.0,2,0.014286,2.0,0.051282,<NA>,NRC con repitencia superior a media + 1 desvia...


In [5]:
display(proposal_actions.head(30))


,rank_prioridad,Descripcion_Materia,nivel_accion,elemento_priorizado,repitencia_asignatura,tasa_repitencia_asignatura,nrc_priorizado,repitencia_nrc_priorizado,tasa_repitencia_nrc_priorizado,criterio_cobertura,cobertura_recomendada_asignatura
0,1,ALGORITMIA Y PROGRAMACION I,Asignatura,ALGORITMIA Y PROGRAMACION I,175,0.603448,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Ambas
1,1,ALGORITMIA Y PROGRAMACION I,NRC,ALGORITMIA Y PROGRAMACION I | NRC 4027,175,0.603448,4027.0,26.0,0.928571,NRC con repitencia superior a media + 1 desvia...,Ambas
2,2,ANÁLISIS DE DATOS EN INGEN I,Asignatura,ANÁLISIS DE DATOS EN INGEN I,64,0.257028,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Ambas
3,2,ANÁLISIS DE DATOS EN INGEN I,NRC,ANÁLISIS DE DATOS EN INGEN I | NRC 2017,64,0.257028,2017.0,14.0,0.4375,NRC con repitencia superior a media + 1 desvia...,Ambas
4,3,REFUERZO ESTATICA,Asignatura,REFUERZO ESTATICA,55,0.591398,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Ambas
5,3,REFUERZO ESTATICA,NRC,REFUERZO ESTATICA | NRC 4121,55,0.591398,4121.0,29.0,1.0,NRC con repitencia superior a media + 1 desvia...,Ambas
6,4,CALCULO I,Asignatura,CALCULO I,55,0.255814,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Ambas
7,4,CALCULO I,NRC,CALCULO I | NRC 2629,55,0.255814,2629.0,19.0,0.513514,NRC con repitencia superior a media + 1 desvia...,Ambas
8,5,OBLIGACIONES I,Asignatura,OBLIGACIONES I,51,0.689189,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Asignatura
9,6,TEORIA DEL DERECHO PROCESAL,Asignatura,TEORIA DEL DERECHO PROCESAL,45,0.661765,<NA>,<NA>,<NA>,Asignatura con repitencia general superior a m...,Asignatura


In [6]:
print(text_visual)


PROPUESTA DE COBERTURA - REPRESENTACION TEXTUAL
Top N seleccionado: 100 | Asignaturas seleccionadas: 100 | NRC: 50 | Asignatura: 44 | Ambas: 6

------------------------------------------------------------------------------------------------
[1] APOYO EN NRC (50)
------------------------------------------------------------------------------------------------
|-- #12 FUNDAMENTO CIENCIAS BASICAS MD [rep_asig=32, tasa_asig=19.8%]
|   `-- NRC 3084 [rep_nrc=9, tasa_nrc=42.9%]

|-- #14 MECANICA DE SOLIDOS [rep_asig=28, tasa_asig=30.8%]
|   `-- NRC 2388 [rep_nrc=13, tasa_nrc=44.8%]

|-- #16 CIENCIA DE LOS MATERIALES [rep_asig=25, tasa_asig=30.5%]
|   `-- NRC 1504 [rep_nrc=6, tasa_nrc=60.0%]

|-- #17 FISICA MECANICA [rep_asig=24, tasa_asig=7.9%]
|   `-- NRC 6106 [rep_nrc=7, tasa_nrc=43.8%]

|-- #18 CALCULO II [rep_asig=24, tasa_asig=6.7%]
|   `-- NRC 2631 [rep_nrc=9, tasa_nrc=8.7%]

|-- #19 ESTATICA [rep_asig=23, tasa_asig=23.5%]
|   `-- NRC 1476 [rep_nrc=14, tasa_nrc=43.8%]

|-- #20 FISICA CAL

In [7]:
saved_paths = save_outputs(
    outputs,
    output_dir=output_dir,
    output_prefix=DEFAULT_OUTPUT_PREFIX,
)

display(
    pd.DataFrame(
        {
            'archivo': list(saved_paths.keys()),
            'ruta': [str(path) for path in saved_paths.values()],
        }
    )
)


,archivo,ruta
0,enriched_df,Resultados_Modelo_v2\propuestas_resultados_v2\...
1,subject_summary,Resultados_Modelo_v2\propuestas_resultados_v2\...
2,nrc_summary,Resultados_Modelo_v2\propuestas_resultados_v2\...
3,proposal,Resultados_Modelo_v2\propuestas_resultados_v2\...
4,proposal_actions,Resultados_Modelo_v2\propuestas_resultados_v2\...
5,flow_visual,Resultados_Modelo_v2\propuestas_resultados_v2\...
6,text_visual,Resultados_Modelo_v2\propuestas_resultados_v2\...
